# 03 — Model: LR Baseline + Fairlearn ThresholdOptimizer

**Project H17 — Fair AI Hiring with Bias Audit.** We train `fair_hiring.models.train_baseline` (LR on the engineered design matrix) and the `train_postprocessed` Fairlearn-style ThresholdOptimizer wrapper. If `fairlearn` isn't available, we provide a per-group threshold fallback that mirrors equalised-odds intuition.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                              roc_curve, classification_report)

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from fair_hiring.models import (train_baseline, train_postprocessed,
                                  fairness_audit, save)
df = pd.read_parquet('../data/processed/resumes.parquet')
len(df)

## 1. Stratified train / test split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=11,
                                      stratify=df['hire_label'])
print(f'train: {len(train_df):,}   test: {len(test_df):,}')
print(f'train positive rate: {train_df["hire_label"].mean():.3f}')

## 2. Train baseline LR

In [ ]:
baseline = train_baseline(train_df)
y_test = test_df['hire_label'].values
y_pred = baseline.predict(test_df)
print(classification_report(y_test, y_pred, digits=3))

## 3. Calibration: ROC + PR curves

In [ ]:
y_proba = baseline.predict_proba(test_df)[:, 1]
auc = roc_auc_score(y_test, y_proba)
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, lw=2, label=f'AUC={auc:.3f}', color='#1f77b4')
ax.plot([0, 1], [0, 1], color='gray', ls=':')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Baseline LR — ROC')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Pre-audit on baseline — gender

In [ ]:
audit_b = fairness_audit(test_df.reset_index(drop=True), y_pred, sensitive_col='gender')
print(audit_b.round(3))
print(f'recall_gap={audit_b.attrs["recall_gap"]:.3f}  '
      f'sel_rate_gap={audit_b.attrs["selection_rate_gap"]:.3f}  '
      f'fpr_gap={audit_b.attrs["fpr_gap"]:.3f}')

## 5. Try Fairlearn ThresholdOptimizer (with per-group threshold fallback)

In [ ]:
pp = train_postprocessed(train_df, baseline, sensitive_col='gender')
if pp is not None:
    print('Using Fairlearn ThresholdOptimizer')
    y_pred_pp = pp.predict(test_df, sensitive_features=test_df['gender'].values)
else:
    print('Fairlearn not available — using stdlib per-group threshold fallback')
    # Fallback: tune the LR threshold per group on the train set so each group's
    # selection rate matches the cohort.
    train_proba = baseline.predict_proba(train_df)[:, 1]
    target = train_df['hire_label'].mean()
    thr_by_g = {}
    for g in train_df['gender'].unique():
        mask = (train_df['gender'].values == g)
        # threshold such that fraction predicted positive equals target
        thr_by_g[g] = float(np.quantile(train_proba[mask], 1 - target))
    print('per-group thresholds:', thr_by_g)
    test_proba = baseline.predict_proba(test_df)[:, 1]
    y_pred_pp = np.array([
        int(p >= thr_by_g[g]) for p, g in zip(test_proba, test_df['gender'].values)
    ])

## 6. Post-audit on the postprocessor / fallback

In [ ]:
audit_p = fairness_audit(test_df.reset_index(drop=True), y_pred_pp, sensitive_col='gender')
print(audit_p.round(3))
print(f'recall_gap={audit_p.attrs["recall_gap"]:.3f}  '
      f'sel_rate_gap={audit_p.attrs["selection_rate_gap"]:.3f}  '
      f'fpr_gap={audit_p.attrs["fpr_gap"]:.3f}')

## 7. Visualise gap shrinkage

In [ ]:
rows = []
for which, audit in [('baseline', audit_b), ('postprocessed', audit_p)]:
    rows.append(dict(model=which, recall_gap=audit.attrs['recall_gap'],
                      sel_rate_gap=audit.attrs['selection_rate_gap'],
                      fpr_gap=audit.attrs['fpr_gap']))
gaps = pd.DataFrame(rows)
long = gaps.melt(id_vars='model', var_name='metric', value_name='gap')
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=long, x='metric', y='gap', hue='model',
            palette=['#fb6a4a', '#1f77b4'], ax=ax)
ax.axhline(0.05, color='black', ls=':', label='deployment gate')
ax.set_title('Per-group gaps — baseline vs postprocessed')
ax.legend(); plt.tight_layout(); plt.show()

## 8. Per-group selection rate — bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, audit, title in [(axes[0], audit_b, 'baseline'), (axes[1], audit_p, 'postprocessed')]:
    sub = audit.copy()
    sub['group_str'] = sub['group'].astype(str)
    long = sub.melt(id_vars='group_str', value_vars=['recall', 'fpr', 'selection_rate'],
                     var_name='metric', value_name='value')
    sns.barplot(data=long, x='metric', y='value', hue='group_str', palette='Set2', ax=ax)
    ax.set_title(title); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

## 9. Audit on a *second* protected attribute — nationality_group

In [ ]:
aud_nat = fairness_audit(test_df.reset_index(drop=True), y_pred, sensitive_col='nationality_group')
print(aud_nat.round(3))
print(f'recall_gap={aud_nat.attrs["recall_gap"]:.3f}  '
      f'sel_rate_gap={aud_nat.attrs["selection_rate_gap"]:.3f}')

## 10. LR coefficient inspection — does the model lean on the proxy?

In [ ]:
from fair_hiring.features import NUMERIC_FEATURES, CATEGORICAL_FEATURES, skill_columns
skills = skill_columns(df)
feat_names = NUMERIC_FEATURES + skills + ['edu_Bachelor', 'edu_Master', 'edu_PhD']
coef = baseline.named_steps['lr'].coef_[0]
rank = np.argsort(-np.abs(coef))[:10]
print('top-10 LR features by |coef|:')
for i in rank:
    print(f'  {feat_names[i] if i < len(feat_names) else f"f{i}":35s}  coef={coef[i]:+.3f}')

## 11. Save artifacts

In [ ]:
save(baseline, 'baseline_lr.joblib')
if pp is not None:
    save(pp, 'fairlearn_eqodds.joblib')
print('saved baseline + (optional) postprocessor')

## 12. Modelling notes
- Baseline LR has good headline AUC but a real gap on per-group selection rate / TPR / FPR — exactly because the model leans on `prior_employer_tier`, a gender-correlated proxy.
- The Fairlearn ThresholdOptimizer (or the stdlib per-group threshold fallback) shrinks the gaps without retraining the underlying classifier.
- The audit on a second attribute (nationality_group) is the right hygiene step — fixing one axis can leak into another.